# Учись Машина Учись / Learn Machine Learn

<img src="data/lml.png" width=200>

Онлайн-лекции Ильи С. Елисеева: применение методов машинного обучения в анализе данных.

- Канал в Telegram: https://t.me/learn_machine_learn
- YouTube: https://www.youtube.com/channel/UCCwDwKatNitBCVAJajremMQ
- VK: https://vk.com/learn_machine_learn
- GitHub: https://github.com/easyise/learn_machine_learn

---



## 3. Приведение трека в порядок

Для чего мы это делаем?

1. Избавляясь от выбросов и телепортов мы приводим показатели трека к реальным значениям
2. Моделируя пропуски, вызванные телепортациями, мы восстанавливаем пропущенные части трека и восстанавливаем километраж тренировки

Как мы это будем делать?

1. Сначала приводить в порядок пространственные показатели: избавляться от телепортаций
2. Затем выполнять сглаживание трека, приводя в порядок скорости и ускорения, вызванные неточностями работы GPS
3. После этого перегенерируем части трека, которые были испорчены телепортациями - используя прокладывание маршрутов между точками начала телепортов и точками их окончания



In [ ]:
%pip install gpxpy geopy geopandas folium contextily ipywidgets osmnx

In [ ]:
import pandas as pd
import geopandas as gpd

import numpy as np
import matplotlib.pyplot as plt

import contextily as cx 

%load_ext autoreload
%autoreload 2

from utils.geodata import *
from __config__ import *

### 3.1 Очистка с помощью кластерного анализа DBSCAN

Наша задача - определить, какие из кластеров являются "хорошими", а какие - "плохими" (выбросами).

Для этого можно использовать различные эвристики, например:
 - Средняя скорость в кластере должна быть в разумных пределах (например, не выше 50 км/ч для велосипедиста).
 - Среднее ускорение в кластере должно быть в разумных пределах (например, не выше 5 м/с²).
 - Продолжительность кластера должна быть достаточно длинной (например, не менее 1 минуты).
 - Применение триангуляции: прибытие в кластер и отъезд из него должны быть по скорости сопоставимы с передвижением от предыдущего к следующему кластеру

In [ ]:
gdf_ski_bad = gpd.read_file('data/Afternoon_Backcountry_Ski.gpx', 
                        layer="track_points")

fields = ['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2']

gdf_ski_bad = recalculate_metrics(gdf_ski_bad)
gdf_ski_bad[fields].describe()

In [ ]:
# проведем кластерный анализ для выявления телепортов
labels, eps = get_cluster_labels(gdf_ski_bad, min_samples=5*60,
                                include=['time', 'speed_kmh'])
gdf_ski_bad['cluster'] = labels
gdf_ski_bad['cluster'] = gdf_ski_bad['cluster'].astype('category')
gdf_ski_bad['cluster'].value_counts()

In [ ]:
ax = get_with_segments(gdf_ski_bad)[['time', 'segment', 'cluster']] \
    .to_crs(epsg=3857) \
    .plot(
        column="cluster",
        cmap="tab10",
        legend=True,
        markersize=5,
        figsize=(12, 12),
)

ax.set_title("Ski +teleport clusters")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1, alpha=0.5)

ax.grid()
plt.show()

In [ ]:
gdf_bike_bad = gpd.read_file('data/Bike_bad.gpx', 
                        layer="track_points")

gdf_bike_bad = recalculate_metrics(gdf_bike_bad)
gdf_bike_bad[fields].describe()

In [ ]:
labels, eps = get_cluster_labels(gdf_bike_bad, 
                                 min_samples=2*60,
                                 include=['time', 'speed_kmh'])
gdf_bike_bad["cluster"] = labels
gdf_bike_bad["cluster"] = gdf_bike_bad["cluster"].astype("category")
n_clusters = np.unique(labels[labels != -1]).size
noise_points = (labels == -1).sum()
print(f"Найдено кластеров: {n_clusters}, шумовых точек: {noise_points}, eps: {eps:.2f}")

In [ ]:
# стоит ли сначала удалить остановки, а потом уже кластеризовать?
gdf_bike_nonstop = gdf_bike_bad[~gdf_bike_bad['geometry'].geom_equals_exact(gdf_bike_bad['geometry'].shift(1), tolerance=1e-9)].copy()
recalculate_metrics(gdf_bike_nonstop)
gdf_bike_stop = gdf_bike_bad.copy()
display(gdf_bike_stop[fields].describe())
gdf_bike_nonstop[fields].describe()

In [ ]:
labels, eps = get_cluster_labels(gdf_bike_nonstop, 
                                 min_samples=2*60,
                                 include=['time', 'speed_kmh'])
gdf_bike_nonstop["cluster"] = labels
gdf_bike_nonstop["cluster"] = gdf_bike_nonstop["cluster"].astype("category")
n_clusters = np.unique(labels[labels != -1]).size
noise_points = (labels == -1).sum()
print(f"Найдено кластеров: {n_clusters}, шумовых точек: {noise_points}, eps: {eps:.2f}")



In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 10))

gdfs = [gdf_bike_stop, gdf_bike_nonstop]

for i in range(2):
    get_with_segments(gdfs[i])[['time', 'segment', 'cluster']] \
        .to_crs(epsg=3857) \
        .plot(
            column="cluster",
            cmap="turbo",
            legend=True,
            alpha=1,
            markersize=5,
            ax=axs[i],
    )
    axs[i].grid()
    axs[i].set_title("Clustering with stops" if i == 0 else "Clustering without stops")
    cx.add_basemap(axs[i], source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1, alpha=0.5)

plt.show()

In [ ]:
# 1. напишем функцию, которая дает нам геопанду с данными по кластерам: время и координаты входа в кластер и выхода из него
def get_gdf_clusters_info(gdf_clustered: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    def calc_distance(group):
        if len(group) < 2:
            return np.nan
        crs = group.estimate_utm_crs()
        group_ = group.to_crs(crs)
        return group_.distance(group_.shift(1)).sum()

    gdf_cluster_info = (
        gdf_clustered[['time', 'geometry']]
        .groupby(gdf_clustered['cluster'])
        .agg(
            time_entry=('time', 'first'),
            time_exit=('time', 'last'),
            point_entry=('geometry', 'first'),
            point_exit=('geometry', 'last'),
            distance=('geometry', calc_distance),
        )
        .sort_values(by='time_entry')
    )
    gdf_cluster_info['speed_kmh'] = gdf_cluster_info['distance'] / gdf_cluster_info['time_exit'].subtract(gdf_cluster_info['time_entry']).dt.total_seconds() * 3.6
    return gdf_cluster_info

gdf_cluster_info = get_gdf_clusters_info(gdf_ski_bad)
gdf_cluster_info


In [ ]:
gdf_cluster_info = get_gdf_clusters_info(gdf_bike_nonstop)
gdf_cluster_info

In [ ]:
# напишем функцию для расчетов метрик кластеров: скорость входа/выхода/базы, расстояния и времени между кластерами
def recalculate_cluster_metrics(gdf_cluster_info: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    # упорядочиваем по времени входа в кластер
    gdf_cluster_info.sort_values(by='time_entry', inplace=True)

    # добавим колонки с данными о покидании предыдущего кластера
    gdf_cluster_info['time_exit_prev'] = gdf_cluster_info['time_exit'].shift(1)
    gdf_cluster_info['point_exit_prev'] = gdf_cluster_info['point_exit'].shift(1)
    # добавим информацию о точках следующего кластера
    gdf_cluster_info['time_entry_next'] = gdf_cluster_info['time_entry'].shift(-1)
    gdf_cluster_info['point_entry_next'] = gdf_cluster_info['point_entry'].shift(-1)
    
    # посчитаем расстояние и время между кластерами
    gdf_cluster_info['dist_in'] = gdf_cluster_info.apply(lambda row: haversine(
        row.point_exit_prev.y if pd.notnull(row.point_exit_prev) else row.point_entry.y, 
        row.point_exit_prev.x if pd.notnull(row.point_exit_prev) else row.point_entry.x,
        row.point_entry.y, row.point_entry.x
    ), axis=1)
    gdf_cluster_info['time_in'] = (gdf_cluster_info['time_entry'] - gdf_cluster_info['time_exit_prev']).dt.total_seconds()
    gdf_cluster_info['speed_in'] = (gdf_cluster_info['dist_in'] / gdf_cluster_info['time_in'] * 3.6).fillna(0)   
    
    # посчитаем расстояние, время и скорость "на выход" и "по базе"
    gdf_cluster_info['dist_out'] = gdf_cluster_info['dist_in'].shift(-1)
    gdf_cluster_info['time_out'] = gdf_cluster_info['time_in'].shift(-1)
    gdf_cluster_info['speed_out'] = gdf_cluster_info['speed_in'].shift(-1)   
    
    gdf_cluster_info['dist_base'] = gdf_cluster_info.apply(lambda row: haversine(
        row.point_exit_prev.y if pd.notnull(row.point_exit_prev) else row.point_entry.y, 
        row.point_exit_prev.x if pd.notnull(row.point_exit_prev) else row.point_entry.x,
        row.point_entry_next.y if pd.notnull(row.point_entry_next) else row.point_entry.y, 
        row.point_entry_next.x if pd.notnull(row.point_entry_next) else row.point_entry.x
    ), axis=1)
    gdf_cluster_info['time_base'] = (gdf_cluster_info['time_entry_next'] - gdf_cluster_info['time_exit_prev']).dt.total_seconds()
    gdf_cluster_info['speed_base'] = (gdf_cluster_info['dist_base'] / gdf_cluster_info['time_base'] * 3.6)
    
    return gdf_cluster_info

gdf_cluster_info = get_gdf_clusters_info(gdf_bike_stop)
gdf_cluster_info = recalculate_cluster_metrics(gdf_cluster_info)
gdf_cluster_info[['time_entry', 'time_exit', 'time_in', 'time_out', 'time_base', 'speed_in','speed_out', 'speed_base']]

In [ ]:
# напишем функцию, которая возвращает список кластеров для удаления по заданным критериям
def identify_bad_clusters(gdf_clustered: gpd.GeoDataFrame,
                            v_lim=70) -> list:
    gdf_cluster_info = get_gdf_clusters_info(gdf_clustered)
    gdf_cluster_info = recalculate_cluster_metrics(gdf_cluster_info)

    bad_clusters = []
    reasons = {}

    # 1. шум (-1) - проверяем, если он пересекается по времени с кластерами, удаляем
    noise_mask = gdf_cluster_info.index == -1
    noise_entry = gdf_cluster_info[noise_mask]['time_entry'].min()
    noise_exit = gdf_cluster_info[noise_mask]['time_exit'].max()
    overlapping_clusters = gdf_cluster_info[~noise_mask][
        ( (gdf_cluster_info['time_entry'] <= noise_exit) &
          (gdf_cluster_info['time_exit'] >= noise_entry) ) 
    ].index.tolist()
    if overlapping_clusters:
        gdf_cluster_info.drop(index=[-1], inplace=True)
        bad_clusters.extend([-1])
        reasons = { -1: "overlapping noise cluster" }

    # 2. удаляем вложенные по времени кластеры итеративно, пока таковых не останется
    while True:
        gdf_cluster_info = recalculate_cluster_metrics(gdf_cluster_info)
        time_in = (gdf_cluster_info['time_entry'] - gdf_cluster_info['time_exit_prev']).dt.total_seconds()
        if gdf_cluster_info[time_in < 0].empty:
            break
        nested_cluster = gdf_cluster_info[time_in < 0].index[0]
        bad_clusters.extend([int(nested_cluster)])
        reasons[int(nested_cluster)] = "nested cluster"
        gdf_cluster_info.drop(index=[nested_cluster], inplace=True)
    

    # 3. вычислим метрики входа-выхода-базы для каждого кластера
    while True:
        gdf_cluster_info = recalculate_cluster_metrics(gdf_cluster_info)
        # вычислим коэффициент триангуляции
        max_speed = gdf_cluster_info[['speed_in', 'speed_out']].max(axis=1)
        gdf_cluster_info['triang_diff'] = (gdf_cluster_info['speed_base'] / max_speed).fillna(1)
        triang_diff_mask = (
            ( (gdf_cluster_info['speed_in'] > v_lim) &
              (gdf_cluster_info['speed_out'] > v_lim) ) &
            (gdf_cluster_info['triang_diff'] < 0.8)
        )
        triang_bad_clusters = gdf_cluster_info[triang_diff_mask].index.tolist()
        # display(gdf_cluster_info[['speed_in', 'speed_out', 'speed_base', 'triang_diff']])
        if not triang_bad_clusters:
            break
        to_remove = triang_bad_clusters[0]
        gdf_cluster_info.drop(index=[to_remove], inplace=True)
        bad_clusters.extend([to_remove])         
        reasons[int(to_remove)] = "triangulation difference"
        
    return bad_clusters, reasons  # уникальные значения

bad_clusters_stop, reasons = identify_bad_clusters(gdf_bike_stop)
bad_clusters_nonstop, reasons_nonstop = identify_bad_clusters(gdf_bike_nonstop)
print(bad_clusters_stop, bad_clusters_nonstop)
display(reasons)
display(reasons_nonstop)

In [ ]:
# отобразим плохие кластеры на карте
fig, axs = plt.subplots(1, 2, figsize=(20, 10))

gdfs = [gdf_bike_stop, gdf_bike_nonstop]
masks = [gdf_bike_stop['cluster'].isin(bad_clusters_stop), gdf_bike_nonstop['cluster'].isin(bad_clusters_nonstop)]

for i in range(2):
    gdfs[i]['suspicious_clusters'] = masks[i]
    get_with_segments(gdfs[i])[['time', 'segment', 'suspicious_clusters']] \
        .to_crs(epsg=3857) \
        .plot(
            column="suspicious_clusters",
            cmap="plasma",
            legend=True,
            alpha=1,
            markersize=5,
            ax=axs[i],
    )
    axs[i].grid()
    axs[i].set_title("Bad clusters with stops" if i == 0 else "Bad clusters without stops")
    cx.add_basemap(axs[i], source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1, alpha=0.5)

plt.show()

In [ ]:
# собственно, очистка
gdf_bike_clus_clean = gdf_bike_nonstop[~gdf_bike_nonstop['suspicious_clusters']].copy()

gdf_bike_clus_clean = recalculate_metrics(gdf_bike_clus_clean)

gdf_bike_clus_clean[['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2', 'acc_g', 'geometry']].describe()




In [ ]:
# визуализируем очищенный трек с сегментами
ax = get_with_segments(gdf_bike_clus_clean)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="turbo",
        vmax=80,
        legend=True,
        markersize=5,
        figsize=(10, 8),
)

ax.set_title("Cleaned track with segments")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1, alpha=0.5)

ax.grid()
plt.show()

### 3.2 Очистка с помощью Z-оценок

Выполним расчет Z-оценок для скоростей и ускорений в треке, чтобы выявить выбросы. Выбросами будут считаться точки, у которых Z-оценка превышает определенный порог (например, 5). Эти точки можно удалить из трека, так как они скорее всего являются результатом ошибок GPS или других аномалий.

In [ ]:
z_thresh = 6

gdf_bike_clean_z, v_rob, a_rob = get_z_scores(gdf_bike_clus_clean, z_thresh)
print("Робастные пороги для скорости: {:.2f} км/ч, для ускорения: {:.2f} м/с^2".format(v_rob, a_rob))
gdf_bike_clean_z[['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2', 'acc_g', 'geometry', 'speed_kmh_z', 'acc_m_per_s2_z']].describe()

In [ ]:
gdf_bike_clean_z['speed_kmh_z_fail'].sum()

In [ ]:
# Очистка трека от выбросов при помощи z-оценки
clean_mask = ~gdf_bike_clean_z['speed_kmh_z_fail']

gdf1 = gdf_bike_clean_z[clean_mask].copy()

recalculate_metrics(gdf1)

gdf1[['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2', 'acc_g', 'geometry']].describe()

In [ ]:
gdf2, v_rob, a_rob = get_z_scores(gdf1, z_thresh)
print("Робастные пороги для скорости: {:.2f} км/ч, для ускорения: {:.2f} м/с^2".format(v_rob, a_rob))
print("Количество выбросов по скорости после второй итерации: {}".format(gdf2['speed_kmh_z_fail'].sum()))


In [ ]:
# есть функция, которая будет итеративно удалять выбросы по z-оценке
def cleanup_by_zscore_iter(gdf, z_thresh=6, 
                        outliers_ratio=0.02, 
                        max_outliers=100,
                        max_iterations=10):
    gdf_ret = gdf.copy()
    for iteration in range(max_iterations):
        gdf_ret_z, v_rob, a_rob = get_z_scores(gdf_ret, z_thresh)
        n_removed = gdf_ret_z['speed_kmh_z_fail'].sum()
        print(f"Iteration {iteration+1}: removed {n_removed} points")
        if n_removed / len(gdf_ret) <= outliers_ratio or n_removed <= max_outliers:
            print(f"Iteration {iteration+1}: not removed {n_removed} points")
            break
        gdf_ret = gdf_ret_z[~gdf_ret_z['speed_kmh_z_fail']].copy()
        recalculate_metrics(gdf_ret)
    return gdf_ret, v_rob, a_rob

gdf_bike_iter_clean, v_iter_rob, a_iter_rob = cleanup_by_zscore_iter(gdf_bike_clus_clean, 
                                                                     z_thresh=z_thresh, 
                                                                     outliers_ratio=0.00, 
                                                                     max_outliers=1,
                                                                     max_iterations=100)
recalculate_metrics(gdf_bike_iter_clean)
print("Final cleaned track length: {:.2f} m".format(gdf_bike_iter_clean.dist.sum()))
print("Final robust speed threshold: {:.2f} km/h".format(v_iter_rob))
gdf_bike_iter_clean[['dist', 'speed_kmh', 'acc_m_per_s2']].describe()

In [ ]:
gdf_ = gdf_bike_clus_clean.copy()

# пересчитаем метрики и метрики триангуляции
recalculate_metrics(gdf_)
recalculate_triangulation_metrics(gdf_)
print("Количество точек с проблемами триангуляции: {}".format((gdf_['triang_diff'] < 0.8).sum()))
gdf_[['speed_kmh', 'speed_out', 'speed_base', 'triang_diff']].describe()

# Запускаем очистку в цикле, пока есть выбросы по триангуляции
def cleanup_triang_iter(gdf_, outliers_ratio=0.0, triang_thres=0.8, speed_thresh=30):
    gdf_ = gdf_.copy()
    iteration = 0
    num_outliers_target = int(gdf_.shape[0] * outliers_ratio)
    recalculate_metrics(gdf_)
    recalculate_triangulation_metrics(gdf_)
    while True:
        mask_triang = (((gdf_['triang_diff'] < triang_thres) | (gdf_['speed_base'] > speed_thresh )) 
                       & ( (gdf_['speed_kmh'] > speed_thresh) |
                         (gdf_['speed_out'] > speed_thresh)
                       )
                     )
        num_outliers = mask_triang.sum()
        if num_outliers <= num_outliers_target:
            print(f"Итерация {iteration}: не выбрасываем {num_outliers} точек (<= {num_outliers_target}), конец")
            break
        print(f"Итерация {iteration}: выбрасываем {num_outliers} точек с плохой триангуляцией (> {num_outliers_target})")
        gdf_ = gdf_[~mask_triang]
        recalculate_metrics(gdf_)
        recalculate_triangulation_metrics(gdf_)
        iteration += 1
    return gdf_

gdf_triang_clean = cleanup_triang_iter(gdf_, outliers_ratio=0.0, triang_thres=0.8, speed_thresh=v_rob)
gdf_triang_clean[['dist', 'speed_kmh', 'acc_m_per_s2', 'speed_kmh', 'speed_out', 'speed_base']].describe()

In [ ]:
# визуализируем очищенный трек с сегментами
ax = get_with_segments(gdf_triang_clean)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="viridis",
        vmax=80,
        legend=True,
        markersize=8,
        figsize=(10, 8),
)

ax.set_title("Cleaned track with segments")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1, alpha=0.5)

ax.grid()
plt.show()

In [ ]:
# посмотрим от чего мы избавились (кроме остановок)

# gdf_base = gdf_bike_clus_clean[~(gdf_bike_clus_clean['speed_kmh'] == 0)]
gdf_base = gdf_bike_clus_clean

gdf_base = gdf_base.copy()
recalculate_metrics(gdf_base)
mask_removed = ~gdf_base.index.isin(gdf_triang_clean.index)

ax = get_with_segments(gdf_base)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="viridis",
        vmax=80,
        legend=True,
        markersize=8,
        figsize=(10, 8),
)

gdf_base[mask_removed].to_crs(epsg=3857) \
    .plot(color='red',
        markersize=10,
        ax=ax,
)

ax.set_title("Removed points")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1, alpha=0.5)

ax.grid()
plt.show()




### 3.4 Полная очистка трека

1. Избавиться от кластеров-телепортов, используя DBSCAN и триангуляцию для определения "плохих" кластеров.
2. Посчитать z-порог для скоростей
3. Выполнить очистку трека по триангуляции, удалив точки, которые не соответствуют критериям триангуляции, а также превышения z-порога скоростей на сторонах треугольника
4. Сегментировать трек, разделив его на отрезки между удаленными точками

In [ ]:
gdf_problem = gpd.read_file('data/Bike_bad.gpx', 
                        layer="track_points")

gdf_problem

In [ ]:
import os, sys

def cleanup_track(gdf, z_thresh=6, verbose=False):

    prndev = sys.stdout if verbose else open(os.devnull, 'w')

    gdf_ = gdf.copy()

    # избавимся от остановок
    gdf_ = gdf_[~gdf_['geometry'].geom_equals_exact(gdf_['geometry'].shift(1), tolerance=1e-9)]

    # рассчитаем метрики
    recalculate_metrics(gdf_)
    if verbose:
        display(gdf_[['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2']].describe())
    gdf_, v_rob, _ = get_z_scores(gdf_, z_thresh)

    print("Робастный порог для скорости: {:.2f} км/ч".format(v_rob), file=prndev)

    # выполним кластерный анализ, получим метки
    labels, eps = get_cluster_labels(gdf_, 
                                 min_samples=2*60,
                                 include=['time', 'speed_kmh'])
    gdf_["cluster"] = labels
    n_clusters = np.unique(labels[labels != -1]).size
    noise_points = (labels == -1).sum()
    gdf_["cluster"] = gdf_["cluster"].astype("category")

    print(f"Найдено кластеров: {n_clusters}, шумовых точек: {noise_points}, eps: {eps:.2f}", file=prndev)

    bad_clusters, reasons = identify_bad_clusters(gdf_, v_lim=v_rob)

    if verbose:
        print(f"Identified bad clusters: {bad_clusters}")
        display(reasons)
    gdf_['suspicious_clusters'] = gdf_['cluster'].isin(bad_clusters)
    
    # удалим подозрительные кластеры
    gdf_ = gdf_[~gdf_['suspicious_clusters']].copy()
    
    # итеративно удалим выбросы по триангуляции в 0
    gdf_ = cleanup_triang_iter(gdf_, outliers_ratio=0.0, triang_thres=0.8, speed_thresh=v_rob)
    
    return gdf_

gdf_clean = cleanup_track(gdf_problem, z_thresh=6, verbose=True)
gdf_clean


In [ ]:
ax = get_with_segments(gdf_clean)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="plasma",
        legend=True,
        markersize=5,
        figsize=(12, 12),
)

ax.set_title("Clean track with speed")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1, alpha=0.7)

ax.grid()
plt.show()

In [ ]:
gdf_bad = gpd.read_file('data/Afternoon_Backcountry_Ski.gpx', 
                        layer="track_points")

gdf_clean = cleanup_track(gdf_bad, z_thresh=6, verbose=True)

ax = get_with_segments(gdf_clean)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="plasma",
        legend=True,
        markersize=5,
        figsize=(12, 12),
)

ax.set_title("Clean track with speed")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1, alpha=0.7)

ax.grid()
plt.show()

In [ ]:
gdf_bad = gpd.read_file('data/Vyborg_Repino.gpx', 
                        layer="track_points")

gdf_clean = cleanup_track(gdf_bad, z_thresh=6, verbose=True)

ax = get_with_segments(gdf_clean)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="plasma",
        legend=True,
        markersize=5,
        figsize=(12, 12),
)

ax.set_title("Clean track with speed")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1, alpha=0.7)

ax.grid()
plt.show()

In [ ]:
print("Расстояние по треку после очистки: {:.2f} м".format(gdf_clean['dist'].sum()))

In [ ]:
gdf_bad = gpd.read_file('data/Klyonovo.gpx', 
                        layer="track_points")

recalculate_metrics(gdf_bad)
ax = get_with_segments(gdf_bad)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="plasma",
        vmax=80,
        legend=True,
        markersize=5,
        figsize=(12, 12),
)

ax.set_title("Dirty track with speed")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1, alpha=0.7)

ax.grid()
plt.show()

In [ ]:
gdf_clean = cleanup_track(gdf_bad, z_thresh=6, verbose=True)

ax = get_with_segments(gdf_clean)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="plasma",
        legend=True,
        markersize=5,
        figsize=(12, 8),
)

ax.set_title("Clean track with speed")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik, zoom_adjust=1, alpha=0.7)

ax.grid()
plt.show()

### 3.3 Заполнение пропусков в треке



In [ ]:
gdf_problem = gpd.read_file('data/Afternoon_Backcountry_Ski.gpx', 
                        layer="track_points")

fields = ['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2']

gdf_problem = recalculate_metrics(gdf_problem)
gdf_problem[fields].describe()

In [ ]:
# выполним кластеризацию
labels, eps = get_cluster_labels(gdf_problem, min_samples=2*60)
gdf_problem["cluster"] = labels
gdf_problem["cluster"] = gdf_problem["cluster"].astype("category")
n_clusters = np.unique(labels[labels != -1]).size

print(f"Найдено кластеров: {n_clusters}, шумовых точек: {(labels == -1).sum()}, eps: {eps:.2f}")

# определим плохие кластеры и удалим их
bad_clusters, reasons = identify_bad_clusters(gdf_problem)
print("Плохие кластеры для удаления:", bad_clusters)
print("Причины удаления кластеров:", reasons)

gdf_clean = gdf_problem[~gdf_problem['cluster'].isin(bad_clusters)].copy()
gdf_clean = recalculate_metrics(gdf_clean)
print("Расстояние по треку после очистки по кластерам {:.2f} м".format(gdf_clean.dist.sum()))

In [ ]:
gdf_clean = cleanup_track(gdf_problem, z_thresh=6, verbose=True)
gdf_clean[fields].describe()

In [ ]:
# определим разрывы в треке
gdf_clean[['timedelta_s', 'dist', 'speed_kmh']].describe()

gdf_clean, dist_rob, time_rob = get_z_scores(gdf_clean, z=7, cols=['dist', 'timedelta_s'])
print(f"Робастный порог по расстоянию: {dist_rob:.2f} m, робастный порог по времени: {time_rob:.2f} s")

In [ ]:
print("Всего разрывов по расстоянию:", gdf_clean['dist_z_fail'].sum(), 
      "по времени:", gdf_clean['timedelta_s_z_fail'].sum())

In [ ]:
# by = 'timedelta_s'
by = 'dist'

gdf_clean.sort_values(by=by, ascending=False)[['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'geometry', 'prev_point', by+'_z', by+'_z_fail']].head(20)

In [ ]:
# визуализируем разрывы на карте
ax = gdf_clean[gdf_clean[by+'_z_fail']][['time', 'geometry', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(figsize=(10, 10),color='red', alpha=0.7, markersize=15)
gdf_clean.shift(1)[gdf_clean[by+'_z_fail']][['time', 'geometry', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(color='blue', alpha=0.7, markersize=15, ax=ax)

get_with_segments(gdf_clean)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="turbo",
        vmin=0,
        vmax=80,
        legend=True,
        alpha=0.5,        linewidth=2,
        figsize=(10, 8),
        ax=ax
)


ax.set_title(f"GPX track, cleaned")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
import osmnx as ox

def graph_for_two_points_bbox(lat1, lon1, lat2, lon2, buffer_m=400, network_type="walk"):
    # bbox: north, south, east, west
    buffer_deg = buffer_m / 111320  # приблизительное преобразование метров в градусы
    north = max(lat1, lat2) + buffer_deg
    south = min(lat1, lat2) - buffer_deg
    east  = max(lon1, lon2) + buffer_deg
    west  = min(lon1, lon2) - buffer_deg

    # добавим буфер (в метрах) — osmnx сам корректно расширит bbox
    G = ox.graph_from_bbox(
        (north, south, east, west),
        network_type=network_type,
        simplify=True
    )
    return G

# пример для двух точек
gdf_gap = gdf_clean.sort_values(by=by, ascending=False).iloc[0]
p1 = gdf_gap['prev_point']
p2 = gdf_gap['geometry']

G = graph_for_two_points_bbox(p1.y, p1.x, p2.y, p2.x, buffer_m=400, network_type="walk")
G

In [ ]:
graph_for_two_points_bbox?

In [ ]:
# API_KEY = "123456789abcdef123456789abcdef123456789abcdef123456789abcdef123456789"

import requests

from shapely.geometry import LineString

start = (p1.x, p1.y)  # lon, lat
end   = (p2.x, p2.y)  # lon, lat

url = "https://api.openrouteservice.org/v2/directions/foot-walking/geojson"

body = {
    "coordinates": [start, end],
    "elevation": True,
}

headers = {
    "Authorization": API_KEY,
    "Content-Type": "application/json"
}

r = requests.post(url, json=body, headers=headers)
data = r.json()
data


In [ ]:
data['features'][0]['geometry']['coordinates']

In [ ]:
coords = data['features'][0]['geometry']['coordinates']

gdf_route = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(
        [c[0] for c in coords],
        [c[1] for c in coords],
    ),
    crs="EPSG:4326"
)
gdf_route['ele'] = [c[2] for c in coords]
gdf_route

In [ ]:
# визуализируем разрывы на карте
ax = get_with_segments(gdf_clean)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="turbo",
        vmin=0,
        vmax=80,
        legend=True,
        alpha=0.5,        linewidth=5,
        figsize=(10, 10),
)

get_with_segments(gdf_route)\
    .to_crs(epsg=3857) \
    .plot(
        color='red',
        alpha=0.7, linewidth=5,
        ax=ax
)


ax.set_title(f"GPX track + route from API")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
def attach_time_ele_constant_speed(
    gdf: gpd.GeoDataFrame,
    t_end,
    t_delta,
):
    """
    Назначает время и высоту точкам маршрута при постоянной скорости.

    Параметры:
    ----------
    gdf : GeoDataFrame с Point геометрией (в порядке маршрута)
    t_end : datetime-like
    t_delta : timedelta-like

    ele_start, ele_end : высоты (м)

    Возвращает:
    ----------
    GeoDataFrame с колонками:
    seq, seg_dist_m, cum_dist_m, time, ele, speed_mps
    """
    out = gdf.copy()

    t_start = pd.Timestamp(t_end) - t_delta
    t_end = pd.Timestamp(t_end)

    total_sec = t_delta.total_seconds()
    if total_sec <= 0:
        raise ValueError("t_end должен быть позже t_start")

    # --- автоматический UTM ---
    utm_crs = out.estimate_utm_crs()
    out_m = out.to_crs(utm_crs)

    # --- расстояния ---
    seg = out_m.geometry.distance(out_m.geometry.shift(1)).fillna(0.0)
    cum = seg.cumsum()

    total_dist = float(cum.iloc[-1])
    if total_dist == 0:
        raise ValueError("Нулевая длина маршрута")

    # --- постоянная скорость ---
    speed_mps = total_dist / total_sec

    # --- время ---
    seconds = cum / total_dist * total_sec
    times = t_start + pd.to_timedelta(seconds, unit="s")

    # --- запись ---
    out["seq"] = range(len(out))
    out["seg_dist_m"] = seg.values
    out["cum_dist_m"] = cum.values
    out["time"] = times
    out["speed_mps"] = speed_mps

    out.attrs["utm_crs"] = utm_crs

    return out

gdf_route_w_time = attach_time_ele_constant_speed(
    gdf_route,
    t_end=gdf_gap['time'],
    t_delta=pd.Timedelta(seconds=gdf_gap['timedelta_s']),
)

gdf_route_w_time

In [ ]:
gdf_clean['time_prev'] = gdf_clean['time'].shift(1)
gdf_clean.sort_values(by='timedelta_s', ascending=False)[['time_prev', 'time', 'timedelta_s', 'dist']].head(5)

In [ ]:
gdf_route_w_time.iloc[1:-2]

In [ ]:
# объединим трек с маршрутом
gdf_combined = pd.concat([gdf_clean, gdf_route_w_time.iloc[1:-2]], ignore_index=True)
gdf_combined = gdf_combined.sort_values(by='time').reset_index(drop=True)
gdf_combined = recalculate_metrics(gdf_combined)
gdf_combined[['time', 'ele', 'dist', 'timedelta_s', 'speed_kmh', 'acc_m_per_s2', 'geometry']].describe()

In [ ]:
ax = gdf_combined[['time', 'geometry', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot( 
        column="speed_kmh",
        cmap="turbo",
        vmin=0,
        vmax=80,
        legend=True,
        alpha=0.9,
        markersize=5,
        figsize=(10, 10),
)

ax.set_title("Combined GPX track + API route colored by speed (km/h)")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show()

In [ ]:
ax = get_with_segments(gdf_combined)[['time', 'segment', 'speed_kmh']] \
    .to_crs(epsg=3857) \
    .plot(
        column="speed_kmh",
        cmap="plasma",
        vmin=0,
        vmax=60,
        legend=True,
        alpha=0.9,
        linewidth=5,
        figsize=(10, 10),
)
ax.set_title("Combined GPX track + API route, colored by speed (km/h)")
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
ax.set_axis_off()
plt.show();

In [ ]:
# посчитаем расстояние по треку после очистки и восстановления разрыва
print("Расстояние по треку после очистки и восстановления разрыва {:.2f} м".format(gdf_combined.dist.sum()))

In [ ]:
# теперь выгрузим результат в GPX
from __future__ import annotations

import xml.etree.ElementTree as ET
from pathlib import Path


def gdf_to_gpx(
    gdf: gpd.GeoDataFrame,
    output_file: str | Path,
    track_name: str = "Strava Activity",
    time_col: str = "time",
    ele_col: str = "ele",
    creator: str = "Python GeoPandas",
) -> None:
    """
    Сохраняет GeoDataFrame с точками в GPX 1.1.

    Ожидается, что:
    - geometry содержит Point
    - time_col содержит datetime или строки, приводимые к datetime
    - ele_col содержит высоту в метрах

    Parameters
    ----------
    gdf : geopandas.GeoDataFrame
        Входной GeoDataFrame.
    output_file : str | Path
        Путь к GPX-файлу.
    track_name : str
        Название трека в GPX.
    time_col : str
        Имя колонки со временем.
    ele_col : str
        Имя колонки с высотой.
    creator : str
        Значение атрибута creator в GPX.
    """

    if gdf.empty:
        raise ValueError("GeoDataFrame пустой.")

    required_cols = {time_col, ele_col, "geometry"}
    missing = required_cols - set(gdf.columns)
    if missing:
        raise ValueError(f"Отсутствуют колонки: {missing}")

    if gdf.crs is None:
        raise ValueError("У GeoDataFrame не задан CRS. Нужен CRS, чтобы перевести координаты в WGS84.")

    # Работаем с копией
    data = gdf.copy()

    # Проверяем, что геометрии - точки
    geom_types = set(data.geometry.geom_type.dropna().unique())
    if not geom_types.issubset({"Point"}):
        raise ValueError(
            f"Ожидаются только Point-геометрии, а получены: {geom_types}"
        )

    # Приводим время
    data[time_col] = pd.to_datetime(data[time_col], utc=True, errors="coerce")
    if data[time_col].isna().any():
        raise ValueError(f"Не удалось преобразовать часть значений в колонке '{time_col}' в datetime.")

    # Сортируем по времени
    data = data.sort_values(time_col).reset_index(drop=True)

    # Переводим в WGS84, если нужно
    if data.crs.to_epsg() != 4326:
        data = data.to_crs(4326)

    # GPX namespace
    ns = "http://www.topografix.com/GPX/1/1"
    ET.register_namespace("", ns)

    gpx = ET.Element(
        f"{{{ns}}}gpx",
        attrib={
            "version": "1.1",
            "creator": creator,
        },
    )

    trk = ET.SubElement(gpx, f"{{{ns}}}trk")
    name_el = ET.SubElement(trk, f"{{{ns}}}name")
    name_el.text = track_name

    trkseg = ET.SubElement(trk, f"{{{ns}}}trkseg")

    for _, row in data.iterrows():
        pt = row.geometry
        lat = pt.y
        lon = pt.x
        ele = row[ele_col]
        t = row[time_col]

        trkpt = ET.SubElement(
            trkseg,
            f"{{{ns}}}trkpt",
            attrib={
                "lat": f"{lat:.8f}",
                "lon": f"{lon:.8f}",
            },
        )

        ele_el = ET.SubElement(trkpt, f"{{{ns}}}ele")
        ele_el.text = f"{float(ele):.3f}"

        time_el = ET.SubElement(trkpt, f"{{{ns}}}time")
        time_el.text = t.strftime("%Y-%m-%dT%H:%M:%SZ")

    tree = ET.ElementTree(gpx)
    output_file = Path(output_file)
    tree.write(output_file, encoding="utf-8", xml_declaration=True)

gdf_to_gpx(gdf_combined, "data/Afternoon_Backcountry_Ski_ok.gpx", track_name="Afternoon Backcountry Ski - Cleaned", creator="GeoPandas GPX Exporter")